# Microsoft GraphRAG — Deep-Dive Native Implementation

## From Knowledge Graphs to Hierarchical Community Summarization

<img src="../images/Microsoft_GraphRag.svg" alt="Microsoft GraphRAG" style="width:100%; height:auto;">

This notebook provides a **complete, from-scratch implementation** of the GraphRAG pipeline
introduced in *"From Local to Global: A Graph RAG Approach to Query-Focused Summarization"*
(Edge et al., Microsoft Research, 2024).

**Stack**: Qwen3-8B (4-bit NF4) · BGE-base-en-v1.5 · FAISS · NetworkX · Pure Python
— no `graphrag` package, no LangChain, no LlamaIndex, no OpenAI API.

## 1. What is Microsoft GraphRAG?

Standard RAG splits documents into chunks, embeds them, and retrieves the most similar
chunks to a query. This works for **local fact-lookup questions** ("What is X?"), but
fails for **global sensemaking questions** that require synthesizing scattered information
("What are the main themes in this dataset?").

Microsoft's **GraphRAG** (Edge et al., 2024) addresses this by building an intermediate
**knowledge graph**, applying **community detection** to identify clusters of densely-
connected entities, then **summarizing each community** with the LLM. At query time,
two search modes are available:

| | Standard RAG | GraphRAG Local | GraphRAG Global |
|---|---|---|---|
| **Retrieval unit** | Text chunks | Entity neighborhoods + community summaries | All community summaries |
| **Strength** | Specific factual recall | Entity-centric questions | Holistic / thematic questions |
| **Weakness** | Misses cross-document connections | Still entity-bound | High indexing cost |

### The Pipeline at a Glance

```
Source Text → Chunking → Entity/Relationship Extraction (LLM)
    → Graph Construction (NetworkX) → Community Detection (Louvain)
    → Community Summarization (LLM) → Hierarchical Summary Tree
                                        │
                            ┌───────────┴───────────┐
                      Local Search              Global Search
                  (entity → community          (map-reduce over
                   → generate)                  all summaries)
```

The key insight is that community summaries act as **pre-computed, structured abstractions**
of the corpus — enabling the LLM to reason over the *entire* dataset without exceeding
its context window.

## 2. Environment Setup

In [ ]:
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy networkx

In [ ]:
import torch, gc, json, re, textwrap, warnings, time
import numpy as np
import networkx as nx
from collections import defaultdict
from typing import List, Dict, Tuple, Optional

warnings.filterwarnings("ignore")
print(f"PyTorch {torch.__version__} — CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CACHE_DIR = "/content/drive/MyDrive/models"
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Model cache directory: {CACHE_DIR}")

### 2.1 Load Qwen3-8B (4-bit NF4)

GraphRAG is **LLM-intensive** — entity extraction, community summarization, and answer
generation all require LLM calls. We load with 4-bit NF4 quantization via `bitsandbytes`
to fit in a single T4/L4 GPU.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} with 4-bit NF4 quantization...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=CACHE_DIR,
)
print(f"Model loaded. Memory: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
def generate(prompt: str, max_new_tokens: int = 1024, temperature: float = 0.6) -> str:
    """Generate text with thinking disabled for structured outputs."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.95,
            do_sample=True,
        )
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

test_response = generate("What is a knowledge graph? Answer in one sentence.")
print(f"LLM test: {test_response}")

### 2.2 Load Embedding Model (BGE-base-en-v1.5)

For vector retrieval we use `BAAI/bge-base-en-v1.5`, a compact 768-dim model. We'll use it
for both the standard RAG baseline and local search entity matching.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL_ID = "BAAI/bge-base-en-v1.5"
embed_model = SentenceTransformer(EMBED_MODEL_ID, cache_folder=CACHE_DIR)
print(f"Embedding model loaded: {EMBED_MODEL_ID}")
print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

test_emb = embed_model.encode(["knowledge graph"], normalize_embeddings=True)
print(f"Test embedding shape: {test_emb.shape}, norm: {np.linalg.norm(test_emb[0]):.4f}")

## 3. Synthetic Knowledge Base

We create a rich fictional dataset about the **NovaTech ecosystem** — a technology
conglomerate with multiple subsidiaries, products, people, and events. This gives us
dense entity relationships to showcase GraphRAG's strengths.

The corpus is designed so that **local questions** ("Who founded Helios Energy?") can be
answered from specific chunks, while **global questions** ("What are the main business
themes across NovaTech?") require synthesizing information scattered across many chunks.

In [ ]:
DOCUMENTS = [
    """NovaTech Industries was founded in 2010 by Dr. Elena Vasquez in Austin, Texas.
Starting as a small AI research lab, NovaTech grew rapidly through strategic acquisitions
and organic innovation. By 2018, the company had expanded into four major divisions:
Quantum Computing (NovaTech Quantum), Renewable Energy (Helios Energy), Biomedical
Devices (VitaCore Medical), and Cloud Infrastructure (SkyGrid Cloud). Dr. Vasquez served
as CEO until 2022, when she transitioned to the role of Chief Science Officer. Marcus
Chen, previously the COO, succeeded her as CEO. NovaTech's corporate headquarters remain
in Austin, with research campuses in Zurich, Singapore, and São Paulo.""",

    """Helios Energy, the renewable energy division of NovaTech Industries, was established
in 2015 under the leadership of Dr. Priya Sharma. The division focuses on next-generation
solar panel technology and grid-scale energy storage. Their flagship product, the SolarWeave
panel, uses perovskite-silicon tandem cells to achieve 33% efficiency — a record for
commercial panels. Helios Energy operates manufacturing plants in Phoenix, Arizona and
Rajasthan, India. In 2023, Helios partnered with the European Space Agency (ESA) on the
Orbital Solar Initiative, an ambitious project to beam solar energy from space to ground
receivers. Dr. Sharma reports directly to Marcus Chen and sits on the NovaTech board.""",

    """VitaCore Medical is NovaTech's biomedical division, led by Dr. James Okafor since
its founding in 2016. VitaCore specializes in neural interface devices and AI-driven
diagnostics. Their NeuroLink implant, approved by the FDA in 2023, allows paralyzed
patients to control prosthetic limbs through thought alone. The device uses a custom
ASIC chip designed in collaboration with NovaTech Quantum's hardware team. VitaCore
operates clinical research centers in Boston and Munich. Dr. Okafor previously worked
at Johns Hopkins and holds 47 patents in biomedical engineering. VitaCore's annual
revenue reached $2.1 billion in 2024, making it NovaTech's fastest-growing division.""",

    """NovaTech Quantum, the quantum computing arm of NovaTech Industries, was launched
in 2017 by Dr. Liang Wei, a former researcher at Google's quantum lab. The division
developed the Orion-7 quantum processor, a 127-qubit superconducting chip that achieved
quantum advantage on specific optimization problems in 2023. NovaTech Quantum operates
a cryogenics lab in Zurich and a fabrication facility in Austin. The division collaborates
closely with SkyGrid Cloud to offer quantum-as-a-service through SkyGrid's data centers.
Dr. Wei's team also designed the custom ASIC used in VitaCore's NeuroLink implant,
demonstrating the cross-divisional synergy that defines NovaTech's strategy.""",

    """SkyGrid Cloud is NovaTech's cloud infrastructure platform, launched in 2018 under
the direction of Sarah Kim, former VP of Engineering at Amazon Web Services. SkyGrid
provides enterprise cloud computing, AI model hosting, and quantum-as-a-service (in
partnership with NovaTech Quantum). SkyGrid's flagship product, CloudMesh, offers a
unified API for managing hybrid cloud deployments across major providers. By 2024,
SkyGrid had 15,000 enterprise customers and operated 12 data centers worldwide. Sarah
Kim's aggressive pricing strategy has positioned SkyGrid as the fastest-growing cloud
provider in Asia-Pacific markets, with regional headquarters in Singapore.""",

    """In September 2024, NovaTech Industries announced the Convergence Initiative, a
$5 billion cross-divisional program to develop integrated quantum-AI-medical solutions.
The initiative pairs NovaTech Quantum's Orion-7 processor with VitaCore's NeuroLink
technology to create next-generation brain-computer interfaces. Marcus Chen described
it as 'the most ambitious project in NovaTech's history.' The initiative also includes
a partnership with Stanford University and the Max Planck Institute for Neuroscience.
Separately, Helios Energy's Orbital Solar Initiative with ESA reached a milestone when
a prototype beamed 10 kW of power from low Earth orbit to a ground station in Spain.""",

    """NovaTech Industries reported consolidated revenue of $18.7 billion for fiscal year
2024, up 23% from the prior year. SkyGrid Cloud was the largest revenue contributor at
$7.2B, followed by Helios Energy ($4.5B), VitaCore Medical ($2.1B), and NovaTech Quantum
($1.8B). Corporate and licensing accounted for the remaining $3.1B. Analysts noted that
NovaTech's cross-divisional synergies — such as quantum chips in medical devices and
quantum-as-a-service through cloud infrastructure — create a competitive moat that
pure-play competitors struggle to match. Dr. Vasquez's original vision of 'convergent
technology platforms' appears to be validated by the financial results.""",

    """NovaTech's leadership team reflects its multidisciplinary approach. CEO Marcus Chen
holds an MBA from Wharton and a PhD in Operations Research from MIT. Dr. Elena Vasquez,
now CSO, earned her doctorate in Computer Science from Stanford and was named to TIME's
100 Most Influential People in 2021. Dr. Priya Sharma (Helios Energy) published over 120
papers on photovoltaics before joining NovaTech. Dr. James Okafor (VitaCore) is a member
of the National Academy of Engineering. Dr. Liang Wei (NovaTech Quantum) was awarded the
Breakthrough Prize in Fundamental Physics in 2022. Sarah Kim (SkyGrid) was recognized by
Forbes as one of the Top 50 Women in Tech. NovaTech employs 34,000 people globally and
was ranked #3 on Glassdoor's Best Places to Work in 2024.""",

    """NovaTech faces competition from established tech giants and specialized startups.
In quantum computing, IBM's Eagle processor and Google's Sycamore compete with the Orion-7.
In cloud infrastructure, SkyGrid Cloud competes with AWS, Azure, and Google Cloud, though
its quantum-as-a-service offering is unique. Helios Energy's SolarWeave panels compete
with First Solar and SunPower, but the Orbital Solar Initiative has no direct competitor.
VitaCore's NeuroLink competes with Neuralink (Elon Musk) and Synchron, though VitaCore's
FDA approval gives it a regulatory advantage. Industry analysts describe NovaTech as a
'mini-conglomerate with startup energy' — its ability to move quickly across divisions
is seen as both its greatest strength and a potential risk if coordination falters.""",

    """Looking ahead to 2025-2027, NovaTech has outlined three strategic priorities.
First, scaling the Convergence Initiative to clinical trials for quantum-enhanced
brain-computer interfaces by 2026. Second, expanding SkyGrid Cloud to 25 data centers
and launching a sovereign cloud offering for government customers. Third, commercializing
Helios Energy's orbital solar technology, with a target of 1 MW orbital capacity by 2027.
Dr. Vasquez has also hinted at a fifth division focused on advanced materials, potentially
through the acquisition of a graphene manufacturing startup. NovaTech's board approved a
$3 billion R&D budget for 2025, the largest in the company's history, signaling confidence
in the convergent technology strategy."""
]

print(f"Knowledge base: {len(DOCUMENTS)} documents")
for i, doc in enumerate(DOCUMENTS):
    print(f"  Doc {i}: {len(doc)} chars — {doc[:70].strip()}...")

## 4. The GraphRAG Indexing Pipeline

### 4.1 Text Chunking

Chunks serve as the **unit of entity extraction** — the LLM processes one chunk at a time
to identify entities and relationships. We use sentence-boundary-aware chunking with overlap.

In [ ]:
def chunk_text(text: str, chunk_size: int = 600, overlap: int = 100) -> List[str]:
    """Split text into overlapping chunks at sentence boundaries."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if end < len(text):
            last_period = chunk.rfind('.')
            if last_period > chunk_size * 0.5:
                end = start + last_period + 1
                chunk = text[start:end]
        chunks.append(chunk.strip())
        start = end - overlap
    return chunks

all_chunks = []
chunk_to_doc = {}

for doc_idx, doc in enumerate(DOCUMENTS):
    for chunk in chunk_text(doc):
        chunk_to_doc[len(all_chunks)] = doc_idx
        all_chunks.append(chunk)

print(f"Total chunks: {len(all_chunks)}")
print(f"\nExample chunk 0:")
print(textwrap.fill(all_chunks[0], width=90))

### 4.2 Entity & Relationship Extraction

This is the **core innovation** of GraphRAG. Instead of just embedding text chunks, we
use the LLM to extract **structured knowledge** — entities (people, orgs, products,
locations, events) and relationships between them.

The extraction prompt produces **structured JSON** that we parse programmatically. The
original paper uses a "gleaning" technique with multiple passes to improve recall; we
implement a single-pass version here.

In [ ]:
ENTITY_EXTRACTION_PROMPT = """You are an expert at extracting structured knowledge from text.

Given the following text, extract ALL entities and relationships.

ENTITY TYPES: person, organization, product, location, event, technology, concept

For each entity provide: name (title case), type, description (brief).
For each relationship provide: source, target, description.

Return ONLY valid JSON in this exact format (no other text):
{
  "entities": [{"name": "...", "type": "...", "description": "..."}],
  "relationships": [{"source": "...", "target": "...", "description": "..."}]
}

TEXT:
"""

def extract_entities_and_relationships(chunk: str) -> Dict:
    """Extract entities and relationships from a text chunk using the LLM."""
    prompt = ENTITY_EXTRACTION_PROMPT + chunk
    response = generate(prompt, max_new_tokens=1024, temperature=0.3)
    try:
        json_match = re.search(r'\{[\s\S]*\}', response)
        if json_match:
            result = json.loads(json_match.group())
            if "entities" not in result: result["entities"] = []
            if "relationships" not in result: result["relationships"] = []
            return result
    except json.JSONDecodeError:
        pass
    return {"entities": [], "relationships": []}

# Test on first chunk
print("Testing extraction on chunk 0...")
print(f"Chunk: {all_chunks[0][:150]}...\n")
test_extraction = extract_entities_and_relationships(all_chunks[0])
print(f"Entities ({len(test_extraction['entities'])}):")
for e in test_extraction['entities']:
    print(f"  [{e.get('type','?')}] {e.get('name','?')}: {e.get('description','?')[:80]}")
print(f"\nRelationships ({len(test_extraction['relationships'])}):")
for r in test_extraction['relationships']:
    print(f"  {r.get('source','?')} → {r.get('target','?')}: {r.get('description','?')[:80]}")

### 4.2.1 Running Extraction Across All Chunks

This is the **most expensive** indexing step. Each chunk requires an LLM call. In
production, Microsoft parallelizes these calls; we process sequentially here.

In [ ]:
print("Extracting entities and relationships from all chunks...")
print("=" * 60)

all_entities = []
all_relationships = []

for i, chunk in enumerate(all_chunks):
    result = extract_entities_and_relationships(chunk)
    for entity in result.get("entities", []):
        all_entities.append((entity, i))
    for rel in result.get("relationships", []):
        all_relationships.append((rel, i))
    print(f"  Chunk {i:2d}: {len(result.get('entities',[]))} entities, {len(result.get('relationships',[]))} relationships")

print(f"\n{'='*60}")
print(f"Total raw entities: {len(all_entities)}")
print(f"Total raw relationships: {len(all_relationships)}")

### 4.3 Entity Resolution & Deduplication

Raw extraction often produces duplicate or variant entity names ("NovaTech" vs "NovaTech
Industries"). We merge these using **normalized string matching** with description merging.
The full GraphRAG pipeline uses embedding similarity + LLM verification for this step.

In [ ]:
def normalize_entity_name(name: str) -> str:
    return re.sub(r'[^a-z0-9 ]', '', name.lower()).strip()

entity_map = {}

for entity, chunk_idx in all_entities:
    name = entity.get("name", "").strip()
    if not name: continue
    norm = normalize_entity_name(name)
    if not norm: continue

    if norm not in entity_map:
        entity_map[norm] = {"name": name, "type": entity.get("type", "unknown"),
                            "descriptions": [], "chunk_indices": set()}
    if len(name) > len(entity_map[norm]["name"]):
        entity_map[norm]["name"] = name
    desc = entity.get("description", "")
    if desc and desc not in entity_map[norm]["descriptions"]:
        entity_map[norm]["descriptions"].append(desc)
    entity_map[norm]["chunk_indices"].add(chunk_idx)

for norm, info in entity_map.items():
    info["description"] = " | ".join(info["descriptions"][:3])
    info["chunk_indices"] = sorted(info["chunk_indices"])

print(f"Deduplicated entities: {len(entity_map)} (from {len(all_entities)} raw)")
print(f"\nEntity catalog:")
for norm, info in sorted(entity_map.items(), key=lambda x: x[1]['name']):
    print(f"  [{info['type']:12s}] {info['name']:30s} chunks={info['chunk_indices']}")

## 5. Graph Construction

Each **deduplicated entity** becomes a node and each **extracted relationship** becomes
an edge. Entity descriptions, types, and relationship descriptions are stored as
node/edge attributes. This graph is the foundation for community detection — the step
that distinguishes GraphRAG from standard RAG.

In [ ]:
G = nx.Graph()

for norm, info in entity_map.items():
    G.add_node(info["name"], type=info["type"], description=info["description"],
               norm_name=norm, chunk_indices=info["chunk_indices"])

norm_to_canonical = {info["norm_name"]: info["name"]
                     for info in entity_map.values()}

edges_added = edges_skipped = 0
for rel, chunk_idx in all_relationships:
    src = norm_to_canonical.get(normalize_entity_name(rel.get("source","")))
    tgt = norm_to_canonical.get(normalize_entity_name(rel.get("target","")))
    desc = rel.get("description", "")
    if src and tgt and src != tgt:
        if G.has_edge(src, tgt):
            old = G[src][tgt].get("description","")
            if desc and desc not in old:
                G[src][tgt]["description"] = old + " | " + desc
            G[src][tgt]["weight"] = G[src][tgt].get("weight",1) + 1
        else:
            G.add_edge(src, tgt, description=desc, weight=1)
        edges_added += 1
    else:
        edges_skipped += 1

print(f"Knowledge Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"  Edges added: {edges_added}, skipped: {edges_skipped}")
print(f"  Density: {nx.density(G):.4f}")
print(f"  Connected components: {nx.number_connected_components(G)}")
print(f"\nTop-10 nodes by degree:")
for node, deg in sorted(G.degree(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {node:35s} [{G.nodes[node].get('type','?'):12s}] degree={deg}")
print(f"\nEdge list:")
for u, v, d in sorted(G.edges(data=True), key=lambda x: x[0]):
    print(f"  {u:30s} ↔ {v:30s} w={d.get('weight',1)} {d.get('description','')[:55]}")

## 6. Community Detection

This step makes GraphRAG unique. We identify **communities** — clusters of densely-
connected entities forming coherent sub-topics. The original paper uses the **Leiden
algorithm** (requires `igraph`); we use NetworkX's built-in **greedy modularity**
optimization, which produces comparable results at this scale.

**Why communities matter:**
1. They provide natural **summarization boundaries**
2. They enable **global search** across all community summaries
3. They reveal the **thematic structure** of the corpus

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities

communities_raw = sorted(greedy_modularity_communities(G, weight='weight'), key=len, reverse=True)

community_map = {}
communities = {}
for cid, members in enumerate(communities_raw):
    communities[cid] = sorted(members)
    for node in members:
        community_map[node] = cid

print(f"Communities detected: {len(communities)}")
print(f"Modularity: {nx.community.modularity(G, communities_raw):.4f}\n")

for cid, members in communities.items():
    types = defaultdict(int)
    for m in members: types[G.nodes[m].get('type','?')] += 1
    tstr = ", ".join(f"{v} {k}" for k,v in sorted(types.items(), key=lambda x:-x[1]))
    print(f"Community {cid} ({len(members)} members — {tstr}):")
    for m in members:
        print(f"    {m:35s} degree={G.degree(m)}")
    print()

## 7. Community Summarization

For each community we generate an **LLM summary** capturing key entities, relationships,
and themes. These summaries are the **pre-computed knowledge** that powers both local and
global search. The original paper creates summaries at multiple hierarchy levels using
Leiden's resolution parameter; we implement single-level summarization.

In [ ]:
COMMUNITY_SUMMARY_PROMPT = """You are an expert analyst. Given the entities and relationships
forming a thematic community in a knowledge graph, write a comprehensive summary.

Include: main theme, key entities and roles, important relationships, notable facts/numbers.

ENTITIES:
{entities}

RELATIONSHIPS:
{relationships}

Write a clear, informative summary in 3-5 paragraphs:"""

def summarize_community(cid: int) -> str:
    members = communities[cid]
    ent_lines = [f"- {n} ({G.nodes[n].get('type','?')}): {G.nodes[n].get('description','N/A')}"
                 for n in members]
    rel_lines = [f"- {u} → {v}: {d.get('description','related')}"
                 for u,v,d in G.edges(data=True) if u in members and v in members]
    prompt = COMMUNITY_SUMMARY_PROMPT.format(
        entities="\n".join(ent_lines) or "None",
        relationships="\n".join(rel_lines) or "None")
    return generate(prompt, max_new_tokens=512, temperature=0.5)

community_summaries = {}
print("Generating community summaries...")
print("=" * 60)
for cid in communities:
    print(f"\nSummarizing community {cid} ({len(communities[cid])} members)...")
    community_summaries[cid] = summarize_community(cid)
    print(f"  Preview: {community_summaries[cid][:200]}...")

print(f"\n{'='*60}")
print(f"Generated {len(community_summaries)} community summaries.")

In [ ]:
for cid, summary in community_summaries.items():
    members = communities[cid]
    print(f"\n{'='*80}")
    print(f"COMMUNITY {cid}: {', '.join(members[:5])}{'...' if len(members)>5 else ''}")
    print(f"{'='*80}")
    print(textwrap.fill(summary, width=90))

## 8. Building FAISS Indices

We build two indices:
1. **Chunk index** — standard RAG-style retrieval over text chunks (baseline)
2. **Entity index** — over entity names + descriptions (for local search entity matching)

Both use cosine similarity via inner product on L2-normalized BGE embeddings.

In [ ]:
import faiss

print("Encoding text chunks...")
chunk_embeddings = embed_model.encode(all_chunks, normalize_embeddings=True, show_progress_bar=True)
chunk_embeddings = np.array(chunk_embeddings, dtype=np.float32)
dim = chunk_embeddings.shape[1]
chunk_index = faiss.IndexFlatIP(dim)
chunk_index.add(chunk_embeddings)
print(f"Chunk FAISS index: {chunk_index.ntotal} vectors, dim={dim}")

entity_texts = []
entity_names = []
for node in G.nodes():
    entity_texts.append(f"{node}: {G.nodes[node].get('description','')}")
    entity_names.append(node)

print("\nEncoding entity descriptions...")
entity_embeddings = embed_model.encode(entity_texts, normalize_embeddings=True, show_progress_bar=True)
entity_embeddings = np.array(entity_embeddings, dtype=np.float32)
entity_index = faiss.IndexFlatIP(dim)
entity_index.add(entity_embeddings)
print(f"Entity FAISS index: {entity_index.ntotal} vectors, dim={dim}")

## 9. Standard RAG Baseline

Before implementing GraphRAG search modes, we build a **standard RAG baseline** — simple
chunk retrieval + generation — for comparison.

In [ ]:
def standard_rag(query: str, top_k: int = 3) -> str:
    """Standard RAG: embed query → retrieve top-k chunks → generate."""
    qe = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = chunk_index.search(qe, top_k)
    ctx = "\n\n".join(f"[Chunk {idx}, score={s:.3f}]:\n{all_chunks[idx]}"
                       for s, idx in zip(scores[0], indices[0]))
    prompt = f"""Answer the question using the context. Be specific and detailed.

CONTEXT:
{ctx}

QUESTION: {query}

ANSWER:"""
    return generate(prompt, max_new_tokens=512, temperature=0.5)

test_q = "Who founded NovaTech Industries?"
print(f"Query: {test_q}\n")
print("Standard RAG answer:")
print(textwrap.fill(standard_rag(test_q), width=90))

## 10. GraphRAG Local Search

Local search handles **entity-centric questions**:

1. **Embed query** → find most relevant entities via FAISS
2. **Expand to communities** → retrieve community summaries for matched entities
3. **Gather supporting chunks** → also retrieve top-k original text chunks
4. **Generate** → combine community summaries + chunks into LLM context

The advantage: community summaries provide **structured context** about entity
relationships that might be spread across many individual chunks.

In [ ]:
def graphrag_local_search(query: str, top_k_entities: int = 5, top_k_chunks: int = 2) -> str:
    """GraphRAG local: entity matching → community context → generate."""
    qe = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)

    # Step 1: Find relevant entities
    scores, indices = entity_index.search(qe, top_k_entities)
    matched_entities = []
    matched_communities = set()
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(entity_names):
            ent = entity_names[idx]
            cid = community_map.get(ent)
            matched_entities.append((ent, score, cid))
            if cid is not None: matched_communities.add(cid)

    # Step 2: Community summaries
    comm_ctx = [f"[Community {cid} — {', '.join(communities[cid][:4])}...]:\n{community_summaries[cid]}"
                for cid in matched_communities if cid in community_summaries]

    # Step 3: Text chunks
    cs, ci = chunk_index.search(qe, top_k_chunks)
    chunk_ctx = [f"[Chunk {idx}]:\n{all_chunks[idx]}" for _, idx in zip(cs[0], ci[0])]

    full_ctx = "\n\n".join(comm_ctx + chunk_ctx)
    prompt = f"""You are answering a question using knowledge graph-enhanced context.
The context includes community summaries (structured entity cluster overviews)
and relevant text chunks.

CONTEXT:
{full_ctx}

QUESTION: {query}

Provide a detailed, specific answer:"""
    return generate(prompt, max_new_tokens=512, temperature=0.5), matched_entities, matched_communities

test_q = "Who founded NovaTech Industries and what is their background?"
print(f"Query: {test_q}\n")
answer, entities, comms = graphrag_local_search(test_q)
print("Matched entities:")
for ent, score, cid in entities:
    print(f"  {ent:35s} score={score:.3f}  community={cid}")
print(f"Communities consulted: {sorted(comms)}\n")
print("Local Search answer:")
print(textwrap.fill(answer, width=90))

## 11. GraphRAG Global Search

Global search handles **holistic, thematic questions** via **map-reduce**:

### Map Phase
For each community summary, generate a **partial answer** rated for relevance (0-100).

### Reduce Phase
Collect all partial answers, sort by relevance, synthesize into a **comprehensive final answer**.

This ensures **every community** gets a chance to contribute — preventing the "lost in
the middle" problem and enabling true cross-corpus synthesis.

In [ ]:
MAP_PROMPT = """Based on this community summary, provide any relevant information for the question.
Rate your response's relevance (0-100).

COMMUNITY SUMMARY:
{summary}

QUESTION: {question}

Return ONLY valid JSON: {{"answer": "...", "relevance": 0-100}}
If irrelevant, set relevance to 0 and answer to "N/A"."""

def global_search_map(query: str) -> List[Dict]:
    """Map phase: partial answers from each community."""
    results = []
    for cid, summary in community_summaries.items():
        resp = generate(MAP_PROMPT.format(summary=summary, question=query),
                        max_new_tokens=300, temperature=0.3)
        try:
            m = re.search(r'\{[\s\S]*\}', resp)
            if m:
                r = json.loads(m.group())
                results.append({"community_id": cid, "answer": r.get("answer","N/A"),
                                "relevance": int(r.get("relevance",0))})
                continue
        except (json.JSONDecodeError, ValueError):
            pass
        results.append({"community_id": cid, "answer": resp[:300], "relevance": 50})
    return sorted(results, key=lambda x: x["relevance"], reverse=True)

test_q = "What are the main business areas and themes across NovaTech?"
print(f"Query: {test_q}\n")
print(f"Map phase — querying {len(community_summaries)} communities...")
partials = global_search_map(test_q)
for pa in partials:
    print(f"  Community {pa['community_id']} (relevance={pa['relevance']}): {pa['answer'][:120]}...")

In [ ]:
REDUCE_PROMPT = """Synthesize the partial answers below into a comprehensive answer.

PARTIAL ANSWERS:
{partial_answers}

QUESTION: {question}

Provide a comprehensive, well-structured answer:"""

def global_search_reduce(query: str, partials: List[Dict]) -> str:
    relevant = [p for p in partials if p["relevance"] > 10] or partials[:3]
    pa_text = "\n\n".join(f"[Community {p['community_id']}, rel={p['relevance']}]:\n{p['answer']}"
                           for p in relevant)
    return generate(REDUCE_PROMPT.format(partial_answers=pa_text, question=query),
                    max_new_tokens=768, temperature=0.5)

def graphrag_global_search(query: str) -> str:
    """Complete global search: map + reduce."""
    partials = global_search_map(query)
    return global_search_reduce(query, partials), partials

print(f"Running full global search for: {test_q}\n")
answer, partials = graphrag_global_search(test_q)
print("Global Search answer:")
print(textwrap.fill(answer, width=90))

## 12. Complete Pipeline Demo

We wrap everything into a clean pipeline and run end-to-end queries comparing all three
search modes: Standard RAG, GraphRAG Local, and GraphRAG Global.

In [ ]:
class GraphRAGPipeline:
    def __init__(self):
        self.graph = G
        self.communities = communities
        self.community_summaries = community_summaries
        print(f"GraphRAG Pipeline: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, "
              f"{len(communities)} communities, {len(all_chunks)} chunks")

    def query(self, question: str, mode: str = "local") -> str:
        if mode == "local":
            answer, _, _ = graphrag_local_search(question)
            return answer
        elif mode == "global":
            answer, _ = graphrag_global_search(question)
            return answer
        raise ValueError(f"Unknown mode: {mode}")

pipeline = GraphRAGPipeline()

In [ ]:
def compare_search(query: str, label: str):
    """Run a query through all three search modes and print results."""
    print(f"\n{'#'*90}")
    print(f"# {label}: {query}")
    print(f"{'#'*90}")

    print("\n--- Standard RAG ---")
    print(textwrap.fill(standard_rag(query), width=90))

    print("\n--- GraphRAG Local Search ---")
    ans, ents, comms = graphrag_local_search(query)
    print(f"Entities: {[e[0] for e in ents[:3]]}, Communities: {sorted(comms)}")
    print(textwrap.fill(ans, width=90))

    print("\n--- GraphRAG Global Search ---")
    ans, partials = graphrag_global_search(query)
    rel = [p['community_id'] for p in partials if p['relevance'] > 30]
    print(f"Relevant communities: {rel}")
    print(textwrap.fill(ans, width=90))

# Q1: Entity-specific
compare_search("What is the NeuroLink implant and who developed it?", "Q1 — Entity-specific")

# Q2: Cross-entity relationships
compare_search("How do NovaTech's divisions collaborate with each other?", "Q2 — Cross-entity")

In [ ]:
# Q3: Global thematic (GraphRAG Global should excel)
compare_search(
    "What are the major themes, initiatives, and strategic priorities across the NovaTech ecosystem?",
    "Q3 — Global thematic")

# Q4: Multi-hop reasoning
compare_search(
    "What is the connection between Dr. Liang Wei's quantum work and VitaCore's medical devices?",
    "Q4 — Multi-hop")

In [ ]:
# Q5: Competitive landscape (requires cross-document synthesis)
compare_search(
    "Who are NovaTech's main competitors and what advantages does NovaTech have?",
    "Q5 — Competitive landscape")

## 13. Analysis: Standard RAG vs GraphRAG Local vs Global

| Query Type | Standard RAG | Local Search | Global Search |
|---|---|---|---|
| **Entity-specific** (Q1) | ✅ Good — chunk likely contains answer | ✅ Good — plus community context | ⚠️ Overkill — scans all communities |
| **Cross-entity** (Q2) | ⚠️ May miss connections across chunks | ✅ Good — community summaries capture links | ✅ Good — sees all cross-division themes |
| **Global thematic** (Q3) | ❌ Poor — retrieves only 2-3 chunks | ⚠️ Partial — limited to matched entities | ✅ **Excellent** — map-reduce covers everything |
| **Multi-hop** (Q4) | ⚠️ Depends on chunk co-occurrence | ✅ Good — graph captures connection path | ✅ Good — community may contain full chain |
| **Competitive** (Q5) | ⚠️ Likely finds 1 competitor chunk | ⚠️ Limited entity scope | ✅ Good — aggregates across communities |

**Key takeaways:**
1. **Standard RAG** excels at specific recall but struggles with cross-document synthesis
2. **Local Search** adds structured entity context — ideal for entity-centric questions
3. **Global Search** is the only mode for "what are the themes across everything" — GraphRAG's killer feature

## 14. Graph Analytics & Cost Analysis

In [ ]:
print("Knowledge Graph Statistics")
print("=" * 60)
print(f"Nodes: {G.number_of_nodes()},  Edges: {G.number_of_edges()}")
print(f"Density: {nx.density(G):.4f},  Components: {nx.number_connected_components(G)}")

degrees = [d for _, d in G.degree()]
print(f"Degree — mean: {np.mean(degrees):.1f}, max: {max(degrees)}, min: {min(degrees)}")

print(f"\nTop entities by betweenness centrality:")
for node, c in sorted(nx.betweenness_centrality(G).items(), key=lambda x:-x[1])[:8]:
    print(f"  {node:35s} [{G.nodes[node].get('type','?'):12s}] centrality={c:.3f}")

print(f"\nEntity type distribution:")
tc = defaultdict(int)
for n in G.nodes(): tc[G.nodes[n].get('type','?')] += 1
for t, c in sorted(tc.items(), key=lambda x:-x[1]): print(f"  {t:15s}: {c}")

print(f"\nCommunity sizes:")
for cid, members in communities.items():
    ie = sum(1 for u,v in G.edges() if u in members and v in members)
    print(f"  Community {cid}: {len(members)} nodes, {ie} internal edges")

# Cost analysis
print(f"\n{'='*60}")
print("GraphRAG Cost Profile")
print(f"{'='*60}")
n_c = len(all_chunks)
n_com = len(communities)
print(f"INDEXING: {n_c} extraction + {n_com} summary = {n_c+n_com} LLM calls")
print(f"QUERY: Standard RAG = 1 call | Local = 1 call | Global = {n_com+1} calls")
print(f"\nAt scale (1000 chunks, 50 communities):")
print(f"  Indexing: ~1050 LLM calls")
print(f"  Global search: ~51 LLM calls per query")

## 15. When to Use GraphRAG

### ✅ GraphRAG Shines When:
1. **Large, richly interconnected corpus** — many entities with complex cross-document relationships
2. **Users ask global/thematic questions** — "What are the trends?", "How do things relate?"
3. **Standard RAG fails on synthesis** — information is scattered across many chunks
4. **Corpus is relatively stable** — high indexing cost is amortized over many queries

### ❌ GraphRAG is Overkill When:
1. **Simple factual lookups** — standard RAG suffices for "What is X?"
2. **Small, simple corpora** — graph adds complexity without proportional benefit
3. **Rapidly changing data** — re-indexing the entire graph is expensive
4. **Cost-sensitive applications** — global search uses N LLM calls per query

### Production Considerations
| Aspect | Detail |
|---|---|
| **Indexing cost** | 10-100x more LLM calls than standard RAG |
| **Latency** | Global search: high (sequential community calls), can be parallelized |
| **Incremental updates** | Graph updates + re-community-detection is expensive; not well-addressed in paper |
| **Leiden vs Louvain** | Leiden produces deterministic, higher-quality communities |
| **Hierarchical summaries** | Multiple resolution levels enable adaptive detail (not implemented here) |

## 16. Comparison: RAG Approaches

| Feature | Standard RAG | Graph RAG (KG) | Microsoft GraphRAG |
|---|---|---|---|
| **Retrieval unit** | Text chunks | Entities + triples | Community summaries |
| **Global questions** | ❌ Poor | ⚠️ Limited | ✅ Excellent |
| **Entity questions** | ✅ Good | ✅ Good | ✅ Good (local search) |
| **Indexing cost** | Low | Medium | High |
| **Query cost** | 1 LLM call | 1 LLM call | 1 (local) or N+1 (global) |
| **Key innovation** | Embedding similarity | Structured knowledge | Hierarchical community abstraction |

GraphRAG's key insight: **community detection + summarization** creates a layer of
pre-computed, structured knowledge that bridges local chunk retrieval and global corpus
understanding.

## 17. Synthesis & Conclusion

### What We Built
We implemented the **complete Microsoft GraphRAG pipeline from scratch**:
1. **Entity & Relationship Extraction** — LLM-based structured extraction from text chunks
2. **Knowledge Graph Construction** — NetworkX graph with entity/relationship properties
3. **Community Detection** — Greedy modularity optimization (analogous to Leiden)
4. **Community Summarization** — LLM-generated narrative summaries per community
5. **Local Search** — Entity-centric retrieval augmented with community context
6. **Global Search** — Map-reduce across all community summaries

### The Core Insight
Community summaries are a form of **pre-computed reasoning**. By summarizing entity
clusters *at indexing time*, we answer global questions *at query time* without the LLM
processing the entire corpus in a single pass.

### Trade-offs
| Benefit | Cost |
|---|---|
| Answers global/thematic questions | 10-100x indexing cost |
| Captures cross-document relationships | Complex pipeline to maintain |
| Structured entity knowledge | Sensitive to extraction quality |
| Dual search modes for flexibility | Global search latency scales with communities |

GraphRAG represents a shift from **retrieval-only** to **retrieval + reasoning** at
index time. Communities act as an intermediate representation — more structured than raw
chunks, more flexible than rigid KG schemas — making it powerful for exploratory analysis
of large, unfamiliar corpora.

**Reference**: Edge, D. et al. (2024). *"From Local to Global: A Graph RAG Approach to
Query-Focused Summarization."* arXiv:2404.16130. Microsoft Research.

In [ ]:
print("Pipeline complete!")
print(f"\nFinal statistics:")
print(f"  Documents: {len(DOCUMENTS)}")
print(f"  Chunks: {len(all_chunks)}")
print(f"  Entities: {G.number_of_nodes()}")
print(f"  Relationships: {G.number_of_edges()}")
print(f"  Communities: {len(communities)}")
print(f"  Summaries: {len(community_summaries)}")
print(f"  Queries demonstrated: 5 (local + global + standard RAG)")
print(f"\nAll demo queries completed successfully.")